# 03 — Keras Callbacks & TensorBoard

Reference: [handson-ml3 ch10](https://github.com/ageron/handson-ml3/blob/main/10_neural_nets_with_keras.ipynb) and [tensorflow.org](https://tensorflow.org)

**Runtime → T4 GPU**

| Callback | Purpose |
|---|---|
| `EarlyStopping` | Stop when val metric stops improving |
| `ModelCheckpoint` | Save best weights automatically |
| `ReduceLROnPlateau` | Halve LR on plateau |
| `LearningRateScheduler` | Custom schedule (warmup + cosine) |
| `TensorBoard` | Rich metrics / histograms / images |
| `CSVLogger` | Log metrics to CSV |
| Custom callback | Print LR each epoch |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau,
    LearningRateScheduler, TensorBoard, CSVLogger
)
import os, datetime, math

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0
y_train = y_train.flatten()
y_test  = y_test.flatten()

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

data_aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='aug')

def build_model():
    inp = keras.Input(shape=(32,32,3))
    x   = data_aug(inp)
    for f in [32, 64, 128]:
        x = layers.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.2)(x)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(10, activation='softmax')(x)
    return keras.Model(inp, out)

print('Dataset loaded, model factory ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CALLBACK 1 — EarlyStopping
# ══════════════════════════════════════════════════════════════════════════════
print('=== EarlyStopping ===')
model = build_model()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

es_cb = EarlyStopping(
    monitor='val_accuracy',   # watch val accuracy
    patience=5,               # stop if no improvement for 5 epochs
    min_delta=1e-3,           # minimum change counts as improvement
    restore_best_weights=True # roll back to best checkpoint
)

hist = model.fit(x_train, y_train, epochs=50, batch_size=128,
                 validation_data=(x_test, y_test),
                 callbacks=[es_cb], verbose=0)

print(f'Stopped at epoch {len(hist.history["accuracy"])} (best restored)')
fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(hist.history['accuracy'],     label='train'); axes[0].plot(hist.history['val_accuracy'], label='val')
axes[0].set_title('EarlyStopping — Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(hist.history['loss'],         label='train'); axes[1].plot(hist.history['val_loss'],     label='val')
axes[1].set_title('EarlyStopping — Loss');    axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CALLBACK 2 — ModelCheckpoint
# ══════════════════════════════════════════════════════════════════════════════
print('=== ModelCheckpoint ===')
model2 = build_model()
model2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

checkpoint_cb = ModelCheckpoint(
    filepath='/tmp/best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,       # only overwrite when val_accuracy improves
    save_weights_only=False,   # save full model (architecture + weights)
    verbose=1
)

hist2 = model2.fit(x_train, y_train, epochs=15, batch_size=128,
                   validation_data=(x_test, y_test),
                   callbacks=[checkpoint_cb, EarlyStopping(patience=5, restore_best_weights=True)],
                   verbose=0)

# Reload best model
best_model = keras.models.load_model('/tmp/best_model.keras')
_, acc = best_model.evaluate(x_test, y_test, verbose=0)
print(f'Reloaded best model — test accuracy: {acc:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CALLBACK 3 — ReduceLROnPlateau
# ══════════════════════════════════════════════════════════════════════════════
print('=== ReduceLROnPlateau ===')
model3 = build_model()
model3.compile(optimizer=keras.optimizers.Adam(1e-2),  # start high
               loss='sparse_categorical_crossentropy', metrics=['accuracy'])

reduce_lr_cb = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,     # new_lr = lr * 0.5
    patience=3,
    min_lr=1e-6,
    verbose=1
)

lr_tracker = []
class LRRecorder(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        lr_tracker.append(float(self.model.optimizer.learning_rate))

hist3 = model3.fit(x_train, y_train, epochs=30, batch_size=128,
                   validation_data=(x_test, y_test),
                   callbacks=[reduce_lr_cb, LRRecorder(),
                               EarlyStopping(patience=7, restore_best_weights=True)],
                   verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist3.history['val_accuracy']); axes[0].set_title('ReduceLROnPlateau — Val Accuracy')
axes[0].grid(True, alpha=0.3)
axes[1].plot(lr_tracker, color='orange'); axes[1].set_title('Learning Rate over Epochs')
axes[1].set_yscale('log'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CALLBACK 4 — LearningRateScheduler (warmup + cosine decay)
# ══════════════════════════════════════════════════════════════════════════════
print('=== LearningRateScheduler (Warmup + Cosine) ===')
TOTAL_EPOCHS  = 30
WARMUP_EPOCHS = 5
MAX_LR        = 1e-2
MIN_LR        = 1e-5

def warmup_cosine_schedule(epoch):
    if epoch < WARMUP_EPOCHS:
        return MAX_LR * (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / (TOTAL_EPOCHS - WARMUP_EPOCHS)
    return MIN_LR + 0.5 * (MAX_LR - MIN_LR) * (1 + math.cos(math.pi * progress))

lr_schedule_cb = LearningRateScheduler(warmup_cosine_schedule, verbose=0)

# Visualise the schedule
epochs_range = list(range(TOTAL_EPOCHS))
lr_vals = [warmup_cosine_schedule(e) for e in epochs_range]
plt.figure(figsize=(9,3))
plt.plot(epochs_range, lr_vals, color='steelblue', lw=2)
plt.axvspan(0, WARMUP_EPOCHS, alpha=0.1, color='red', label='warmup')
plt.xlabel('Epoch'); plt.ylabel('Learning Rate'); plt.yscale('log')
plt.title('Warmup + Cosine Decay Schedule'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

model4 = build_model()
model4.compile(optimizer=keras.optimizers.SGD(momentum=0.9),
               loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist4 = model4.fit(x_train, y_train, epochs=TOTAL_EPOCHS, batch_size=128,
                   validation_data=(x_test, y_test),
                   callbacks=[lr_schedule_cb], verbose=0)
print(f'Final val acc: {hist4.history["val_accuracy"][-1]:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CALLBACK 5 — Custom Callback
# ══════════════════════════════════════════════════════════════════════════════
class TrainingMonitor(keras.callbacks.Callback):
    """Custom callback: tracks overfitting gap and prints a warning."""
    def __init__(self, gap_threshold=0.1):
        super().__init__()
        self.gap_threshold = gap_threshold
        self.gaps = []

    def on_epoch_end(self, epoch, logs=None):
        tr_acc = logs.get('accuracy',     0)
        va_acc = logs.get('val_accuracy', 0)
        gap    = tr_acc - va_acc
        self.gaps.append(gap)
        lr = float(self.model.optimizer.learning_rate)
        msg = f'  [Monitor] Epoch {epoch+1:02d} | gap={gap:.3f} | lr={lr:.2e}'
        if gap > self.gap_threshold:
            msg += '  ⚠ OVERFITTING'
        print(msg)

print('=== Custom TrainingMonitor callback ===')
monitor_cb = TrainingMonitor(gap_threshold=0.08)
model5 = build_model()
model5.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist5 = model5.fit(x_train, y_train, epochs=12, batch_size=128,
                   validation_data=(x_test, y_test),
                   callbacks=[monitor_cb], verbose=0)

plt.figure(figsize=(9,3))
plt.plot(monitor_cb.gaps, 'o-', color='crimson')
plt.axhline(monitor_cb.gap_threshold, ls='--', color='gray', label='threshold')
plt.title('Train–Val Accuracy Gap (Overfitting Indicator)')
plt.xlabel('Epoch'); plt.ylabel('Gap'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CALLBACK 6 — CSVLogger
# ══════════════════════════════════════════════════════════════════════════════
print('=== CSVLogger ===')
csv_path = '/tmp/training_log.csv'
csv_cb = keras.callbacks.CSVLogger(csv_path, append=False)

model6 = build_model()
model6.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model6.fit(x_train, y_train, epochs=10, batch_size=128,
           validation_data=(x_test, y_test),
           callbacks=[csv_cb], verbose=0)

df = pd.read_csv(csv_path)
print(df.to_string(index=False))
df.plot(x='epoch', y=['accuracy','val_accuracy'], figsize=(8,4),
        title='Training Log (from CSV)')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CALLBACK 7 — TensorBoard
# ══════════════════════════════════════════════════════════════════════════════
print('=== TensorBoard ===')
log_dir = '/tmp/tb_logs/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')

tb_cb = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1,      # log weight histograms every epoch
    write_graph=True,      # visualise model graph
    write_images=True,     # log model weights as images
    update_freq='epoch',   # flush every epoch
    profile_batch=0        # disable profiler (faster in Colab)
)

model_tb = build_model()
model_tb.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_tb.fit(x_train, y_train, epochs=15, batch_size=128,
             validation_data=(x_test, y_test),
             callbacks=[tb_cb,
                        EarlyStopping(patience=5, restore_best_weights=True)],
             verbose=1)

print(f'\nLogs saved to: {log_dir}')
print('Run the cell below to launch TensorBoard')

In [ ]:
# Launch TensorBoard inline in Colab
%load_ext tensorboard
%tensorboard --logdir /tmp/tb_logs

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BONUS — Custom TensorBoard: log augmented images & confusion matrix
# ══════════════════════════════════════════════════════════════════════════════
import io, itertools
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix_to_image(model, x, y, class_names):
    preds = model.predict(x[:1000], verbose=0).argmax(axis=1)
    cm    = confusion_matrix(y[:1000], preds)
    fig, ax = plt.subplots(figsize=(10,8))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im)
    ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45); ax.set_yticklabels(class_names)
    for i,j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
    plt.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format='png'); buf.seek(0)
    img = tf.image.decode_png(buf.getvalue(), channels=4)[tf.newaxis]
    plt.close(fig)
    return img

cm_img = plot_confusion_matrix_to_image(model_tb, x_test, y_test, CLASS_NAMES)

cm_writer = tf.summary.create_file_writer(log_dir + '/cm')
with cm_writer.as_default():
    tf.summary.image('Confusion Matrix', cm_img, step=0)
cm_writer.flush()
print('Confusion matrix logged to TensorBoard')

In [ ]:
# ── All callbacks together ────────────────────────────────────────────────────
print('=== All Callbacks Combined ===')
full_log_dir = '/tmp/tb_logs/full_' + datetime.datetime.now().strftime('%H%M%S')

callbacks_all = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
    ModelCheckpoint('/tmp/full_best.keras', save_best_only=True, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=0),
    TensorBoard(log_dir=full_log_dir, histogram_freq=1),
    keras.callbacks.CSVLogger('/tmp/full_log.csv'),
    TrainingMonitor(gap_threshold=0.08),
]

final_model = build_model()
final_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])
final_model.fit(x_train, y_train, epochs=40, batch_size=128,
                validation_data=(x_test, y_test),
                callbacks=callbacks_all, verbose=0)

_, test_acc = final_model.evaluate(x_test, y_test, verbose=0)
print(f'\nFinal test accuracy with all callbacks: {test_acc:.4f}')